# 04 — Model Comparison

Final comparison of all models with selection reasoning.

**Production Model:** XGBoost  
**Test MAE:** 21.34  
**Test R²:** 0.6584

In [ ]:
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load comparison results
with open('../models/production/model_comparison_full.json') as f:
    comparison = json.load(f)

models_data = comparison['models']
print(f'Best model: {comparison["best_model"]}')
print(f'Data rows: {comparison["data_rows"]}')
print(f'Features: {comparison["features"]}')

## 1. Overall Performance

In [ ]:
# Comparison table
table = []
for name, data in models_data.items():
    test = data.get('test_metrics', {})
    table.append({
        'Model': name,
        'Test MAE': f'{test.get("mae", 0):.2f}',
        'Test RMSE': f'{test.get("rmse", 0):.2f}',
        'Test R²': f'{test.get("r2", 0):.4f}',
        'Train Time': f'{data.get("train_time", 0):.1f}s',
    })
pd.DataFrame(table)

In [ ]:
# MAE comparison bar chart
model_names = list(models_data.keys())
mae_values = [models_data[m]['test_metrics']['mae'] for m in model_names]
r2_values = [models_data[m]['test_metrics']['r2'] for m in model_names]

fig = make_subplots(rows=1, cols=2, subplot_titles=['Test MAE (Lower is Better)', 'Test R² (Higher is Better)'])

colors = ['#2ecc71' if m == 'XGBoost' else '#3498db' for m in model_names]

fig.add_trace(go.Bar(x=model_names, y=mae_values, marker_color=colors, name='MAE'), row=1, col=1)
fig.add_trace(go.Bar(x=model_names, y=r2_values, marker_color=colors, name='R²'), row=1, col=2)

fig.update_layout(height=400, showlegend=False)
fig.show()

## 2. Per-Horizon Comparison

In [ ]:
horizons = ['24h', '48h', '72h']
horizon_data = []
for h in horizons:
    for name in model_names:
        m = models_data[name]['test_metrics']
        horizon_data.append({
            'Horizon': h,
            'Model': name,
            'MAE': m[f'mae_{h}'],
            'RMSE': m[f'rmse_{h}'],
            'R²': m[f'r2_{h}'],
        })

hdf = pd.DataFrame(horizon_data)
print(hdf.to_string(index=False))

In [ ]:
# Per-horizon MAE bar chart
fig = go.Figure()
for name in model_names:
    h_mae = [hdf[(hdf['Model'] == name) & (hdf['Horizon'] == h)]['MAE'].values[0] for h in horizons]
    fig.add_trace(go.Bar(name=name, x=horizons, y=h_mae))
fig.update_layout(barmode='group', title='MAE by Horizon (Test Set)', yaxis_title='MAE', height=400)
fig.show()

## 3. Selection Reasoning

In [ ]:
print("""=== MODEL SELECTION REASONING ===")
print(f"Production Model: {comparison['best_model']}")
print(f"Test MAE: {models_data['XGBoost']['test_metrics']['mae']:.2f}")
print(f"Test R²: {models_data['XGBoost']['test_metrics']['r2']:.4f}")
print()
print("Why XGBoost:")
print("1. Best Test MAE (21.34) — lowest prediction error")
print("2. Best Test R² (0.6584) — explains most variance")
print("3. Wins on ALL 3 horizons (24h, 48h, 72h)")
print("4. Fast training (9.9s) — suitable for daily retraining")
print("5. Fast inference (0.011ms) — production-ready")
print()
print("Why not others:")
print("- RandomForest: Close but slightly worse on all test metrics")
print("- Ridge: Linear model, cannot capture non-linear patterns")
print("- LSTM: Overfits, worst test performance")
""")